# 02 — Chuẩn hóa text/image embeddings

Pipeline Coveo v1 — mọi output được version hóa và không ghi đè.

In [ ]:
%pip install -q -e ".[coveo]"

In [ ]:
from pathlib import Path
import os, json, yaml

def find_repo():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / 'pyproject.toml').exists(): return root
    raise FileNotFoundError('Không tìm thấy pyproject.toml')

REPO = find_repo()
os.chdir(REPO)
cfg = yaml.safe_load((REPO / 'configs/coveo.yaml').read_text(encoding='utf-8'))
PROFILE = os.getenv('COVEO_PROFILE', cfg['project']['profile'])
print('repo=', REPO, 'profile=', PROFILE)

In [ ]:
from datn.features.coveo import build_coveo_embeddings
processed = REPO / cfg['paths']['processed_dir']
report = build_coveo_embeddings(REPO / cfg['paths']['raw_dir'] / 'sku_to_content.csv',
    processed / 'items.parquet', REPO / cfg['paths']['embeddings_dir'])
display(report)

In [ ]:
import numpy as np
for name in ('text', 'image'):
    x = np.load(REPO / cfg['paths']['embeddings_dir'] / f'{name}_embeddings.npy')
    norms = np.linalg.norm(x[1:], axis=1); present = norms > 0
    print(name, x.shape, 'coverage=', present.mean(), 'mean norm=', norms[present].mean() if present.any() else 0)